# Capítulo 9 · Laboratório 9: seguidores de tendência
*Análise Técnica dos Mercados Financeiros*, 4ª edição, Flávio Lemos, CMT

Dados até **30/09/2026**. Na primeira execução, o notebook baixa os dados do Yahoo Finance na sua máquina e guarda uma cópia local (`dados_cap09.pkl`), que as execuções seguintes reaproveitam. O provedor corrige o histórico (proventos, desdobramentos), por isso seus números podem diferir um pouco dos do livro. Médias, envelopes, Bollinger, Donchian, Keltner, SAR, HiLo, ADX, Aroon e Ichimoku escritos à mão, com pandas. O laboratório confere as contas do texto e testa: quantos fechamentos ficam dentro das Bandas de Bollinger, o que acontece depois do squeeze, cinco seguidores de tendência no Ibovespa (com o MAC de Jake Bernstein), o ADX como filtro, o HiLo na LUNA e os sinais forte, médio e fraco do Ichimoku.

In [ ]:
!pip install -q yfinance

In [ ]:
import logging
import os
import numpy as np
import pandas as pd
import yfinance as yf

logging.getLogger("yfinance").setLevel(logging.CRITICAL)
FIM = "2026-10-01"                      # dados até 30/09/2026
CACHE = "dados_cap09.pkl"
pc = lambda x, k=1: f"{x:.{k}%}".replace(".", ",")
ps = lambda x, k=2: ("+" if x >= 0 else "") + pc(x, k)
nr = lambda x, k=2: f"{x:,.{k}f}".replace(",", "X").replace(".", ",").replace("X", ".")
SERIES = {"^BVSP": "2000-01-01", "PETR4.SA": "2010-01-01", "VALE3.SA": "2010-01-01", "ITUB4.SA": "2010-01-01",
          "ABEV3.SA": "2010-01-01", "BBAS3.SA": "2010-01-01", "LUNC-USD": "2021-06-01"}


def baixa(t, ini):
    d = yf.download(t, start=ini, end=FIM, auto_adjust=True, progress=False)
    if isinstance(d.columns, pd.MultiIndex):
        d.columns = d.columns.get_level_values(0)
    return d.dropna(subset=["Close"])


def confere(baixados):
    """O Yahoo devolve uma tabela vazia quando falha: para aqui, com uma mensagem clara, sem gravar a cópia local."""
    for nome, x in baixados.items():
        if len(x) == 0:
            raise RuntimeError(f"Não consegui baixar {nome} do Yahoo Finance. O provedor pode estar fora do ar ou"
                               " limitando pedidos. Espere alguns minutos e rode a célula de novo.")


D = pd.read_pickle(CACHE) if os.path.exists(CACHE) else {}
for t, ini in SERIES.items():
    if t not in D:
        D[t] = baixa(t, ini)
confere({t: D[t] for t in SERIES})
pd.to_pickle(D, CACHE)

## Parte 1: dados

In [ ]:
def limpa(d, indice=False):
    ok = (d[["Open", "High", "Low", "Close"]] > 0).all(axis=1) & (d["High"] >= d["Low"])
    if not indice:
        ok &= d["Volume"] > 0
    return d[ok].copy()


IBOV = limpa(D["^BVSP"], True)
PETR, VALE, ITUB, ABEV, BBAS = (limpa(D[t]) for t in ("PETR4.SA", "VALE3.SA", "ITUB4.SA", "ABEV3.SA", "BBAS3.SA"))
LUNA = limpa(D["LUNC-USD"], True)
print(f"Parte 1. Ibovespa desde {IBOV.index[0]:%Y}; PETR4, VALE3, ITUB4, ABEV3 e BBAS3 desde {PETR.index[0]:%Y}"
      f" (preços ajustados); último pregão {IBOV.index[-1]:%d/%m/%Y}.")

## Parte 2: os indicadores, escritos à mão

In [ ]:
def mms(c, n):
    return c.rolling(n).mean()


def mme(c, n):
    """MME: começa na MMS dos n primeiros dias e depois soma fator x (fechamento - MME de ontem)."""
    fator = 2 / (n + 1)
    saida = pd.Series(np.nan, c.index)
    if len(c) < n:
        return saida
    valor = c.iloc[:n].mean()
    saida.iloc[n - 1] = valor
    for i in range(n, len(c)):
        valor = valor + fator * (c.iloc[i] - valor)
        saida.iloc[i] = valor
    return saida


def mmvol(d, n):
    """Média ponderada pelo volume: soma de (volume x fechamento) dividida pela soma dos volumes."""
    return (d["Close"] * d["Volume"]).rolling(n).sum() / d["Volume"].rolling(n).sum()


def envelope(c, n=21, pct=0.03):
    m = mms(c, n)
    return m * (1 + pct), m, m * (1 - pct)


def bollinger(c, n=20, k=2.0, ddof=0):
    """Bollinger usa o desvio-padrão populacional (ddof=0); planilhas usam o amostral (ddof=1)."""
    m = mms(c, n)
    dp = c.rolling(n).std(ddof=ddof)
    return m + k * dp, m, m - k * dp


def pct_b(c, n=20, k=2.0):
    sup, _, inf = bollinger(c, n, k)
    return (c - inf) / (sup - inf)


def bandwidth(c, n=20, k=2.0):
    sup, m, inf = bollinger(c, n, k)
    return (sup - inf) / m


def donchian(d, n=20):
    sup, inf = d["High"].rolling(n).max(), d["Low"].rolling(n).min()
    return sup, (sup + inf) / 2, inf


def true_range(d):
    return pd.concat([d["High"] - d["Low"], (d["High"] - d["Close"].shift()).abs(),
                      (d["Low"] - d["Close"].shift()).abs()], axis=1).max(axis=1)


def atr(d, n=14):
    return true_range(d).ewm(alpha=1 / n, adjust=False).mean()          # suavização de Wilder


def keltner_classico(d, n=10):
    """Keltner (1960): média de 10 do preço típico ± média de 10 da amplitude (máxima - mínima)."""
    pt = (d["High"] + d["Low"] + d["Close"]) / 3
    amp = mms(d["High"] - d["Low"], n)
    m = mms(pt, n)
    return m + amp, m, m - amp


def keltner_atr(d, n=20, k=2.5):
    """Versão moderna: média exponencial de 20 ± 2,5 ATR (configuração popularizada por Linda Raschke)."""
    m = d["Close"].ewm(span=n, adjust=False).mean()
    a = atr(d, n)
    return m + k * a, m, m - k * a


def regressao(c):
    x = np.arange(len(c))
    b, a = np.polyfit(x, c.to_numpy(), 1)
    reta = pd.Series(a + b * x, c.index)
    dp = (c - reta).std(ddof=0)
    return reta, dp


def sar(d, passo=0.02, maximo=0.20):
    """SAR parabólico de Wilder: SAR(amanhã) = SAR(hoje) + FA x (PE - SAR(hoje)). PE é o ponto extremo da
    tendência; FA começa em 0,02 e sobe 0,02 a cada novo ponto extremo, até 0,20."""
    h, l = d["High"].to_numpy(), d["Low"].to_numpy()
    out, tend = np.full(len(h), np.nan), np.zeros(len(h), int)
    alta = h[1] >= h[0]
    s, pe, fa = (l[0], h[0], passo) if alta else (h[0], l[0], passo)
    for i in range(1, len(h)):
        s = s + fa * (pe - s)
        if alta:
            s = min(s, l[i - 1], l[i - 2] if i > 1 else l[i - 1])
            if l[i] < s:
                alta, s, pe, fa = False, pe, l[i], passo
            elif h[i] > pe:
                pe, fa = h[i], min(fa + passo, maximo)
        else:
            s = max(s, h[i - 1], h[i - 2] if i > 1 else h[i - 1])
            if h[i] > s:
                alta, s, pe, fa = True, pe, h[i], passo
            elif l[i] < pe:
                pe, fa = l[i], min(fa + passo, maximo)
        out[i], tend[i] = s, 1 if alta else -1
    return pd.Series(out, d.index), pd.Series(tend, d.index)


def hilo(d, n=3):
    """HiLo Activator (Robert Krausz): média das mínimas e das máximas dos últimos n períodos. Vira para baixo
    quando o fechamento fica abaixo da média das mínimas de ontem; vira para cima acima da média das máximas."""
    mh, ml = mms(d["High"], n).shift(), mms(d["Low"], n).shift()
    c = d["Close"]
    estado, ultimo = [], 0
    for ci, a, b in zip(c.to_numpy(), mh.to_numpy(), ml.to_numpy()):
        if not np.isnan(a):
            if ci > a:
                ultimo = 1
            elif ci < b:
                ultimo = -1
        estado.append(ultimo)
    estado = pd.Series(estado, d.index)
    return estado, ml.where(estado == 1, mh)


def adx(d, n=14):
    """ADX de Wilder: +DM, -DM, true range, DI e DX, todos com a suavização de Wilder (fator 1/n)."""
    sobe, desce = d["High"].diff(), -d["Low"].diff()
    mais = pd.Series(np.where((sobe > desce) & (sobe > 0), sobe, 0.0), d.index)
    menos = pd.Series(np.where((desce > sobe) & (desce > 0), desce, 0.0), d.index)
    a = atr(d, n)
    mais_di = 100 * mais.ewm(alpha=1 / n, adjust=False).mean() / a
    menos_di = 100 * menos.ewm(alpha=1 / n, adjust=False).mean() / a
    dx = 100 * (mais_di - menos_di).abs() / (mais_di + menos_di)
    return dx.ewm(alpha=1 / n, adjust=False).mean(), mais_di, menos_di


def adxr(d, n=14):
    a, _, _ = adx(d, n)
    return (a + a.shift(n)) / 2


def aroon(d, n=25):
    cima = d["High"].rolling(n + 1).apply(lambda x: x.argmax(), raw=True) / n * 100
    baixo = d["Low"].rolling(n + 1).apply(lambda x: x.argmin(), raw=True) / n * 100
    return cima, baixo


def ichimoku(d, t=9, k=26, s=52):
    meio = lambda n: (d["High"].rolling(n).max() + d["Low"].rolling(n).min()) / 2
    tenkan, kijun = meio(t), meio(k)
    span_a = ((tenkan + kijun) / 2).shift(k)           # já deslocados para a data em que aparecem no gráfico
    span_b = meio(s).shift(k)
    chikou = d["Close"].shift(-k)
    return tenkan, kijun, span_a, span_b, chikou


def didi(c):
    """Didi Index: médias simples de 3, 8 e 20 (21 na Trader Brasil) divididas pela média de 8."""
    m8 = mms(c, 8)
    return mms(c, 3) / m8, mms(c, 21) / m8


def williams_ad(d):
    """Acumulação/Distribuição de Larry Williams (máxima e mínima verdadeiras, sem volume)."""
    c_ant = d["Close"].shift()
    sobe = d["Close"] - np.minimum(d["Low"], c_ant)
    desce = d["Close"] - np.maximum(d["High"], c_ant)
    return pd.Series(np.where(d["Close"] > c_ant, sobe, np.where(d["Close"] < c_ant, desce, 0.0)), d.index).cumsum()


def mac(d):
    """Moving Average Channel de Jake Bernstein: média simples de 10 das máximas e de 8 das mínimas."""
    return mms(d["High"], 10), mms(d["Low"], 8)

## Parte 3: as contas do texto

In [ ]:
p = pd.Series([11, 12, 13, 14, 15, 16, 17], dtype=float)
print("\nParte 3. As contas do texto")
print(f"   MMS de 5 dias sobre 11 a 17: {', '.join(nr(x, 0) for x in mms(p, 5).dropna())}")
print(f"   Peso do último preço: MME de 10 {pc(2 / 11, 2)}; MME de 20 {pc(2 / 21, 2)} (quase metade, não metade exata)")
c = PETR["Close"].loc["2026"]
certa = mme(c, 10)
errada, v = pd.Series(np.nan, c.index), c.iloc[:10].mean()
m10 = mms(c, 10)
for i in range(10, len(c)):                                # fórmula da 3ª ed.: subtrai a MMS em vez da MME
    v = v + 2 / 11 * (c.iloc[i] - m10.iloc[i - 1])
    errada.iloc[i] = v
print(f"   MME de 10 da PETR4 em {c.index[-1]:%d/%m/%Y}: conta certa R$ {nr(certa.iloc[-1])}; com a MMS no lugar da"
      f" MME de ontem, R$ {nr(errada.iloc[-1])}; ewm do pandas, R$ {nr(c.ewm(span=10, adjust=False).mean().iloc[-1])}")
print(f"   Aroon para cima com máxima há 5 dias num período de 25: {(25 - 5) / 25 * 100:.0f}")

## Parte 4: dentro das bandas

In [ ]:
DENTRO = {}
print("\nParte 4. Fechamentos dentro das Bandas de Bollinger (desde 2010; desvio populacional)")
for nome, d in (("Ibovespa", IBOV.loc["2010":]), ("PETR4", PETR), ("VALE3", VALE), ("ITUB4", ITUB)):
    c = d["Close"]
    linha = []
    for n, k in ((10, 1.9), (20, 2.0), (50, 2.1)):          # ajustes da tabela de Bollinger
        sup, _, inf = bollinger(c, n, k)
        ok = sup.notna()
        linha.append(((c <= sup) & (c >= inf))[ok].mean())
    DENTRO[nome] = linha
    print(f"   {nome:<9} (10; 1,9) {pc(linha[0])} | (20; 2,0) {pc(linha[1])} | (50; 2,1) {pc(linha[2])}")
c = IBOV["Close"]
a0, a1 = bollinger(c, ddof=0), bollinger(c, ddof=1)
print(f"   Ibovespa, (20; 2): bandas com desvio amostral {pc(((a1[0] - a1[2]) / (a0[0] - a0[2])).median() - 1, 1)}"
      f" mais largas que com o populacional")

## Parte 5: squeeze

In [ ]:
BW = bandwidth(c)
SQ = BW <= BW.rolling(125).min()                          # menor largura dos últimos ~6 meses
mov20 = c.shift(-20) / c - 1
sq = mov20[SQ].dropna()
SQUEEZE = {"dias": len(sq), "mov_sq": sq.abs().mean(), "mov_todos": mov20.abs().mean(),
           "largura": (BW.shift(-20)[SQ] / BW[SQ]).median(), "sobe_sq": (sq > 0).mean(),
           "sobe_todos": (mov20.dropna() > 0).mean()}
print(f"\nParte 5. Squeeze no Ibovespa desde 2000 (largura das bandas na mínima de 125 pregões): {SQUEEZE['dias']} dias")
print(f"   Largura das bandas 20 pregões depois: {nr(SQUEEZE['largura'])} vezes a do dia do squeeze")
print(f"   Movimento absoluto em 20 pregões: depois do squeeze {pc(SQUEEZE['mov_sq'])}; num dia qualquer"
      f" {pc(SQUEEZE['mov_todos'])}")
print(f"   Subiu em {pc(SQUEEZE['sobe_sq'], 0)} dos squeezes (num dia qualquer, {pc(SQUEEZE['sobe_todos'], 0)})")

## Parte 6: seguidores de tendência

In [ ]:
def estado(entra, sai):
    pos, lista = 0, []
    for e, s in zip(entra.to_numpy(), sai.to_numpy()):
        if pos == 0 and e:
            pos = 1
        elif pos == 1 and s:
            pos = 0
        lista.append(pos)
    return pd.Series(lista, entra.index)


def mac_sinal(d, com_ad=True):
    """Compra com 2 barras inteiras acima do canal (e A/D acima da sua média de 57); sai com 2 barras inteiras
    abaixo (e A/D abaixo da média). Alvos e saídas parciais não testados."""
    topo, fundo = mac(d)
    acima = (d["Low"] > topo) & (d["Low"].shift() > topo.shift())
    abaixo = (d["High"] < fundo) & (d["High"].shift() < fundo.shift())
    if com_ad:
        ad = williams_ad(d)
        acima &= ad > ad.rolling(57).mean()
        abaixo &= ad < ad.rolling(57).mean()
    return estado(acima, abaixo)


def resultado(c, pos):
    ret = c.pct_change().fillna(0) * pos.shift().fillna(0)
    curva = (1 + ret).cumprod()
    return {"ao ano": curva.iloc[-1] ** (252 / len(c)) - 1, "pior queda": (curva / curva.cummax() - 1).min(),
            "tempo comprado": pos.mean(), "operações": int((pos.diff() == 1).sum()), "curva": curva}


d = IBOV
SISTEMAS = {"Comprar e segurar": pd.Series(1, c.index),
            "Preço acima da MM200": (c > mms(c, 200)).astype(int),
            "Cruzamento MM50/MM200": (mms(c, 50) > mms(c, 200)).astype(int),
            "Donchian 20/10 (inspirado no S1 das tartarugas)": estado(c > d["High"].rolling(20).max().shift(),
                                                                      c < d["Low"].rolling(10).min().shift()),
            "MAC de Bernstein, sem A/D": mac_sinal(d, False),
            "MAC de Bernstein + A/D de Williams (57)": mac_sinal(d)}
RES = {k: resultado(c, v) for k, v in SISTEMAS.items()}
print(f"\nParte 6. Seguidores de tendência no Ibovespa, {c.index[0]:%Y} a {c.index[-1]:%d/%m/%Y} (só comprado, sem custos)")
for k, r in RES.items():
    print(f"   {k:<48} {pc(r['ao ano']):>6} ao ano | pior queda {pc(r['pior queda'], 0):>5} | comprado"
          f" {pc(r['tempo comprado'], 0):>4} | {r['operações']:>3} operações")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 4.5))
for k, r in RES.items():
    ax.plot(r["curva"].index, r["curva"], lw=1, label=f"{k}: {pc(r['ao ano'])} ao ano")
ax.set_yscale("log")
ax.set_title("R$ 1 aplicado no Ibovespa em 2000 (escala log)")
ax.legend(fontsize=7)
plt.show()

## Parte 7: ADX como filtro

In [ ]:
ADX, MAIS, MENOS = adx(d)
rompe = c > d["High"].rolling(20).max().shift()
FILTRO = {}
print("\nParte 7. Rompimentos de máxima de 20 dias no Ibovespa: retorno nos 20 pregões seguintes")
for nome, f in (("ADX abaixo de 20", ADX < 20), ("ADX de 20 a 25", (ADX >= 20) & (ADX <= 25)),
                ("ADX acima de 25", ADX > 25), ("Todos", ADX.notna())):
    x = mov20[rompe & f].dropna()
    FILTRO[nome] = (len(x), x.mean(), (x > 0).mean())
    print(f"   {nome:<17} {len(x):>4} casos | média {ps(x.mean())} | positivos {pc((x > 0).mean(), 0)}")

## Parte 8: HiLo na LUNA

In [ ]:
EST, LINHA_HILO = hilo(LUNA, 3)
virou = EST[(EST == -1) & (EST.shift() == 1)].loc["2022-04-01":"2022-05-08"]
SAIDA_LUNA = virou.index[-1]
topo = LUNA["Close"].loc["2022-03-01":"2022-05-09"]
fundo = LUNA["Close"].loc["2022-05-09":"2022-05-31"].min()
print(f"\nParte 8. HiLo (3) na LUNA: viradas de abril e maio de 2022: "
      + "; ".join(f"{dt:%d/%m} {'venda' if v == -1 else 'compra'}" for dt, v in
                  EST.loc["2022-04-01":"2022-05-09"][EST.diff() != 0].items()))
print(f"   Último sinal de venda antes do colapso em {SAIDA_LUNA:%d/%m/%Y}, a US$"
      f" {nr(LUNA['Close'].loc[SAIDA_LUNA])}; topo de fechamento US$ {nr(topo.max())} em {topo.idxmax():%d/%m/%Y};"
      f" fundo de maio US$ {nr(fundo, 6)} ({pc(fundo / LUNA['Close'].loc[SAIDA_LUNA] - 1, 4)} desde a saída)")
print(f"   Viradas do HiLo (3) em 2021 e 2022: {int((EST.diff() != 0).loc['2021-07':'2022-12'].sum())}")

In [ ]:
import matplotlib.pyplot as plt
j = LUNA.loc["2022-03-01":"2022-05-12"]
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(j.index, j["Close"], color="black", lw=1, label="LUNA (fechamento)")
ax.plot(j.index, LINHA_HILO.loc[j.index], color="gray", lw=1, drawstyle="steps-mid", label="HiLo (3)")
ax.axvline(SAIDA_LUNA, ls="--", color="gray")
ax.legend()
plt.show()

## Parte 9: Ichimoku

In [ ]:
def sinais_ichimoku(d):
    tk, kj, sa, sb, _ = ichimoku(d)
    cruza = (tk > kj) & (tk.shift() <= kj.shift())
    topo_n, fundo_n = np.maximum(sa, sb), np.minimum(sa, sb)
    tipo = np.where(d["Close"] > topo_n, "forte (acima da nuvem)",
                    np.where(d["Close"] < fundo_n, "fraco (abaixo da nuvem)", "médio (dentro da nuvem)"))
    r = d["Close"].shift(-20) / d["Close"] - 1
    ok = cruza & sa.notna() & sb.notna() & r.notna()
    return pd.DataFrame({"tipo": tipo[ok.to_numpy()], "r20": r[ok].to_numpy()})


ICHI = pd.concat([sinais_ichimoku(x.loc["2010":]) for x in (IBOV, PETR, VALE, ITUB, ABEV, BBAS)])
TAB_ICHI = ICHI.groupby("tipo")["r20"].agg(["count", "mean", lambda x: (x > 0).mean()])
TAB_ICHI.columns = ["casos", "média", "positivos"]
print("\nParte 9. Cruzamentos de alta do Tenkan sobre o Kijun (Ibovespa e cinco ações, desde 2010): 20 pregões depois")
for tipo, r in TAB_ICHI.iterrows():
    print(f"   {tipo:<24} {int(r.casos):>4} casos | média {ps(r['média'])} | positivos {pc(r.positivos, 0)}")

## Peça à IA

O Colab tem um assistente de IA. Abra o assistente, cole um destes pedidos e confira a resposta rodando a célula indicada.

1. **Explicar:** "Explique, linha por linha, a função `mme` da Parte 2. Mostre onde ela começa na média simples dos n primeiros dias e onde entra o fator 2 / (n + 1)." Confira com a Parte 3, que imprime o peso de 18,18% da MME de 10 e de 9,52% da MME de 20 e mostra que, na PETR4, a conta certa dá o mesmo valor que a `ewm` do pandas, e a fórmula com a MMS no lugar da MME de ontem dá outro.
2. **Adaptar:** "Crie uma célula nova no fim do notebook que baixe [ATIVO], com o ticker do Yahoo Finance, pelas funções `baixa` e `limpa`, e calcule, como na Parte 4, a porcentagem de fechamentos dentro das Bandas de Bollinger nos ajustes (10; 1,9), (20; 2,0) e (50; 2,1). Use nomes novos de variáveis e não mude a função `bollinger` nem o desvio populacional." Antes de olhar o ativo novo, confira se a Parte 4 continua dando, no Ibovespa, 89,8%, 88,9% e 87,6%, os números da Tabela 9.4.

Mais pedidos deste capítulo (entender e aplicar ao seu gráfico) estão na página do capítulo, no hub do livro.

## Para ir além
1. Na Parte 4, troque o desvio populacional (ddof=0) pelo amostral (ddof=1). Quanto muda o percentual dentro das bandas?
2. Na Parte 6, mude o Donchian para 55/20 (o Sistema 2 das tartarugas). Melhora ou piora no Ibovespa?
3. Na Parte 9, troque o prazo de 20 para 60 pregões. O sinal forte do Ichimoku continua perdendo do fraco?

**Governança:** os resultados deste notebook valem só para os dados e as datas usadas. Mudar regras até o resultado ficar bonito é sobreajuste (*overfitting*): a regra decora o passado e falha no futuro.

━━━━━━━━━━━━━━━━━━━━━━━

⚠️ AVISO LEGAL (DISCLAIMER):

━━━━━━━━━━━━━━━━━━━━━━━

Todo o conteúdo deste livro  é estritamente educacional e informativo. 

Não constitui, em nenhuma hipótese, recomendação de compra ou venda de ativos financeiros. 

Testes passados não são garantias de resultados futuros.

Investimentos envolvem riscos, incluindo a perda total do capital investido. 

Consulte sempre um profissional habilitado antes de tomar qualquer decisão de investimento.